# YOLOv8 Tomato Disease Identification

**Workflow**
1. Mount Google Drive
2. Install dependencies
3. Download dataset from Roboflow
4. Train the model *(skip if already trained)*
5. Load and validate the trained model
6. Run inference with side-by-side visualisation

## 0 - Global Configuration
Edit all paths and hyperparameters here, not scattered across cells.

In [ ]:
# ─────────────────────────────────────────────────
# GLOBAL CONFIG
# ─────────────────────────────────────────────────

# Dataset
ROBOFLOW_WORKSPACE = "group-a9-major"
ROBOFLOW_PROJECT   = "plantseg-whx0t-47j0b"
ROBOFLOW_VERSION   = 3
DATASET_PATH       = "/content/PlantSeg_dataset"

# Model
BASE_MODEL         = "yolov8m.pt"
DRIVE_OUTPUT_DIR   = "/content/drive/MyDrive/YOLOv8_PlantSeg"
RUN_NAME           = "plantseg_yolov8m"
MODEL_PATH         = "/content/best.pt"

# Training hyperparameters
TRAIN_EPOCHS   = 25
TRAIN_PATIENCE = 15
TRAIN_BATCH    = 16
TRAIN_IMGSZ    = 640
TRAIN_SEED     = 42
TRAIN_WORKERS  = 2

# Inference
CONF_THRESHOLD    = 0.25  # minimum detection confidence
DISPLAY_THRESHOLD = 0.80  # minimum confidence for high-confidence badge
MAX_RETRY         = 50    # attempts to find a qualifying sample

## 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2 - Install Dependencies

In [ ]:
%pip install -q ultralytics roboflow

import torch

print("-" * 50)
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
print("-" * 50)
!nvidia-smi

## 3 - Download Dataset from Roboflow

In [ ]:
import os
import shutil
from pathlib import Path
from roboflow import Roboflow
from google.colab import userdata

# Add your key once via: Colab -> Secrets (key icon) -> ROBOFLOW_API_KEY
try:
    api_key = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    import getpass
    api_key = getpass.getpass("Roboflow API key: ")

# Download dataset
rf      = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8")

# Rename to a fixed, predictable path
fixed_path = Path(DATASET_PATH)
src_path   = Path(dataset.location)

if src_path.resolve() != fixed_path.resolve():
    if fixed_path.exists():
        shutil.rmtree(fixed_path)
    shutil.move(str(src_path), str(fixed_path))

dataset.location = str(fixed_path)

print(f"Dataset location : {dataset.location}")
print(f"Contents         : {os.listdir(dataset.location)}")

## 4 - Train the Model
> **Skip this cell** if you already have a trained `best.pt`.

In [ ]:
from ultralytics import YOLO
import os, shutil

model   = YOLO(BASE_MODEL)
results = model.train(
    # Dataset
    data       = os.path.join(DATASET_PATH, "data.yaml"),
    imgsz      = TRAIN_IMGSZ,
    batch      = TRAIN_BATCH,
    cache      = True,

    # Training
    epochs     = TRAIN_EPOCHS,
    patience   = TRAIN_PATIENCE,
    seed       = TRAIN_SEED,
    pretrained = True,
    workers    = TRAIN_WORKERS,

    # Output
    project    = DRIVE_OUTPUT_DIR,
    name       = RUN_NAME,
    exist_ok   = True,
)

# Copy best weights to MODEL_PATH so all later cells find it automatically
trained_best = os.path.join(DRIVE_OUTPUT_DIR, RUN_NAME, "weights", "best.pt")
shutil.copy(trained_best, MODEL_PATH)

# Signal that the model is already loaded — Cell 5 will detect this and skip
_model_trained_this_session = True

print(f"Best weights saved to : {MODEL_PATH}")
print(f"Validation data       : {DATASET_PATH}/data.yaml")
print()
print("✅ Step 5 (model download) will be skipped automatically.")
print("   Run Cell 6 directly for validation, or Cell 7 for inference.")

## 5 - Load and Validate Trained Model
> Downloads `best.pt` from Google Drive, then runs validation on the test split.

In [ ]:
%pip install -q ultralytics gdown

import os
import gdown
from ultralytics import YOLO

# Auto-skip: if Cell 4 already ran this session, best.pt is ready
if globals().get("_model_trained_this_session") and os.path.exists(MODEL_PATH):
    print(f"Using freshly trained model at: {MODEL_PATH}")
    print("Skipping download — jump straight to validation below.")
else:
    GDRIVE_FILE_ID = "10ruXjW7GPaylln-Lt0RJwQ8gOP-4nG_j"
    gdown.download(
        f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}",
        MODEL_PATH,
        quiet=False,
    )
    print(f"Model loaded from: {MODEL_PATH}")

# Validate (always runs)
model   = YOLO(MODEL_PATH)
metrics = model.val(data=f"{DATASET_PATH}/data.yaml")

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")

## 6 - Inference: Side-by-Side Visualisation

In [ ]:
import random
import io
import contextlib
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image as PILImage
from ultralytics import YOLO

# Setup
model      = YOLO(MODEL_PATH)
images_dir = Path(DATASET_PATH) / "test" / "images"
labels_dir = Path(DATASET_PATH) / "test" / "labels"

all_images = [
    f for f in images_dir.iterdir()
    if f.suffix.lower() in {".jpg", ".jpeg", ".png"}
]

if not all_images:
    raise FileNotFoundError(f"No images found in {images_dir}")


def read_ground_truth(image_path: Path) -> str:
    """Return the class name from the first label line, or 'Unknown'."""
    label_path = labels_dir / (image_path.stem + ".txt")
    if label_path.exists():
        parts = label_path.read_text().strip().split()
        if parts:
            return model.names[int(parts[0])]
    return "Unknown"


def run_inference(image_path: Path):
    """Run model inference silently. Returns ultralytics Results object."""
    with (
        contextlib.redirect_stdout(io.StringIO()),
        contextlib.redirect_stderr(io.StringIO()),
    ):
        return model.predict(
            source=str(image_path),
            conf=CONF_THRESHOLD,
            save=False,
            verbose=False,
        )[0]


# Search for a high-confidence correct prediction
result_data = None

for attempt in range(MAX_RETRY):
    img_path = random.choice(all_images)
    actual   = read_ground_truth(img_path)
    res      = run_inference(img_path)

    if not (res.boxes and len(res.boxes)):
        continue

    best_box   = max(res.boxes, key=lambda b: float(b.conf))
    predicted  = model.names[int(best_box.cls)]
    confidence = float(best_box.conf)
    correct    = predicted == actual

    if correct and confidence > DISPLAY_THRESHOLD:
        result_data = dict(
            img_path=img_path, actual=actual, predicted=predicted,
            confidence=confidence, res=res, correct=correct,
        )
        break

# Fallback: show the last sampled image if no qualifying result found
if result_data is None:
    print(f"No high-confidence correct prediction found in {MAX_RETRY} attempts.")
    print("Displaying last sampled image instead.")
    result_data = dict(
        img_path=img_path, actual=actual, predicted=predicted,
        confidence=confidence, res=res, correct=correct,
    )

# Build the visualisation
d          = result_data
orig_img   = PILImage.open(d["img_path"])
pred_img   = PILImage.fromarray(d["res"].plot()[..., ::-1])  # BGR -> RGB
status_txt = "Correct" if d["correct"] else "Incorrect"
conf_color = "#4caf50" if d["confidence"] > DISPLAY_THRESHOLD else "#ff9800"

BG  = "#1e1e1e"
fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=BG)
fig.subplots_adjust(bottom=0.22)

for ax, img, title in zip(axes, [orig_img, pred_img], ["Original", "Prediction"]):
    ax.imshow(img)
    ax.set_title(title, color="white", fontsize=14, fontweight="bold", pad=10)
    ax.set_facecolor(BG)
    ax.axis("off")

summary = (
    f"File       : {d['img_path'].name}\n"
    f"Actual     : {d['actual']}\n"
    f"Predicted  : {d['predicted']}\n"
    f"Confidence : {d['confidence']:.2%}   [{status_txt}]"
)
fig.text(
    0.5, 0.03, summary,
    ha="center", va="bottom",
    fontsize=11, color="white", fontfamily="monospace",
    multialignment="left",
    bbox=dict(boxstyle="round,pad=0.6", facecolor="#2d2d2d",
              edgecolor=conf_color, linewidth=1.5),
)

plt.tight_layout(rect=[0, 0.22, 1, 1])
plt.show()